# EdNet KT3 extract (PRIVATE, on Google Colab)

**Education data: private only.** This notebook downloads EdNet KT3 (Santa TOEIC app, CC BY-NC 4.0, already anonymised) onto Google's temporary computer, turns a random sample of students into one row per answered question, and saves the result to **your private Drive folder `ednet-private/`**. Nothing goes to any public repository; the code lives only in this notebook.

- Measures and predictions: see the project doc `ednet-predictions.md` (written before any data).
- **Student IDs** are replaced by codes made from your secret key (the same one as for Lichess: Colab key icon → `LICHESS_ID_SALT`, or type it when asked). The key is checked against the Lichess fingerprint.
- **Skill-tag tables only:** set `CONTENTS_ONLY = True` (about 2 minutes); it saves `questions.csv` and `lectures.csv` to `ednet-private/` and stops.
- **How to run:** Runtime → Run all, allow Drive access, keep the tab open.
- **First:** keep `TEST_RUN = True` (about 10 minutes: download + 500 students). Then set it to `False` (10,000 students; about 30–60 minutes).
- If the automatic download fails (Google Drive sometimes limits downloads of popular files), open http://bit.ly/ednet-kt3 and http://bit.ly/ednet-content in your browser, click *Add shortcut to Drive* (or download and upload to `My Drive/ednet-private/raw/`), and run again: the notebook then reads them from your Drive.

In [ ]:
# Settings
CONTENTS_ONLY = False   # True: only save EdNet's small question/lecture tables (skill tags) to Drive, then stop
TEST_RUN = True
N_STUDENTS = 500 if TEST_RUN else 10_000   # random students kept (with at least MIN_Q answered questions)
MIN_Q = 100
OUT = "ednet_kt3_answers" + ("_test" if TEST_RUN else "")

In [ ]:
# Tools and private Drive folder
!pip -q install gdown pyarrow
import os, glob, zipfile, requests, gdown
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ednet-private"
os.makedirs(DEST + "/raw", exist_ok=True)
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")

In [ ]:
# Secret key (same as Lichess) for anonymous student codes, checked against the saved fingerprint
import hashlib, hmac, getpass
key = None
try:
    from google.colab import userdata
    key = userdata.get("LICHESS_ID_SALT")
except Exception:
    pass
if not key:
    key = getpass.getpass("Secret key (same as for Lichess): ")
assert len(key) >= 12, "use the same long key as for Lichess"
fp = hashlib.pbkdf2_hmac("sha256", key.encode(), b"lichess-key-check", 200_000).hex()[:6]
fp_file = "/content/drive/MyDrive/compression-learner-data/lichess/key_fingerprint.txt"
if os.path.exists(fp_file):
    saved = open(fp_file).read().strip()
    assert fp == saved, f"STOP: not the same key as for Lichess (fingerprint {fp}, saved {saved})"
    print("key check: same key as Lichess ✓")
else:
    print("no Lichess fingerprint found; key fingerprint", fp)
def student_code(u):
    return "s_" + hmac.new(key.encode(), ("ednet:" + u).encode(), hashlib.sha256).hexdigest()[:12]

In [ ]:
%%writefile ednet_pipeline.py
"""EdNet KT3 -> one row per answered question, with the measures in ednet-predictions.md.

PRIVATE: education data. Run only in Colab / private places; outputs go to a private Drive folder.

Input : EdNet-KT3.zip (one CSV per student: timestamp, action_type, item_id, source, user_answer,
        platform) and EdNet-Contents (questions.csv, lectures.csv).
Output: answers.parquet (one row per answered question) for a random sample of students, and
        answers_events.parquet (one row per explanation or lecture view: which, when, source,
        duration, position in the answer sequence q_idx).
first_answer / t_first_ms: the first letter chosen and how fast (before any change) = impulse;
answer = the final letter submitted.

How actions are read (per student, in time order):
- enter b..  -> a bundle of questions is shown (source, platform); respond q.. -> answer letter
  (the last one before submit counts; n_responds counts changes); submit b.. -> the bundle ends;
  elapsed_ms = submit - enter (shared by the bundle's questions).
- enter/quit e.. -> reading an explanation; reading time after an attempt is credited to that
  question until its next attempt (expl_after_prev_ms).
- enter/quit l.. -> watching a lecture on a skill tag (source archive / recommendation /
  adaptive_offer).
A new session starts after 30 minutes without actions.
"""
import argparse
import collections
import io
import math
import random
import sys
import zipfile

import pandas as pd

SESSION_GAP_MS = 30 * 60 * 1000
MIN_LETTERS = 20          # earlier answers needed before the favourite letter counts


def load_contents(questions_csv, lectures_csv):
    q = pd.read_csv(questions_csv)
    l = pd.read_csv(lectures_csv)
    qinfo = {}
    for r in q.itertuples(index=False):
        tags = [t for t in str(r.tags).split(";") if t not in ("", "-1", "nan")]
        qinfo[r.question_id] = (r.correct_answer, int(r.part), tuple(tags), r.explanation_id, r.bundle_id)
    linfo = {}
    for r in l.itertuples(index=False):
        tags = [t for t in str(r.tags).split(";") if t not in ("", "-1", "nan")]
        linfo[r.lecture_id] = tuple(tags)
    expl_to_q = collections.defaultdict(list)
    for qid, (_, _, _, eid, _) in qinfo.items():
        expl_to_q[eid].append(qid)
    return qinfo, linfo, expl_to_q


def process_student(user, df, qinfo, linfo, expl_to_q, events=None):
    """df: one student's actions sorted by time. Returns a list of row dicts.
    If `events` is a list, explanation and lecture views are appended to it (one dict each)."""
    rows = []
    t0 = None
    last_ts = None
    session_idx, session_pos, session_start = -1, 0, None
    bundle = None                      # (bundle_id, enter_ts, source, platform, answers{q: [letter, n]})
    expl_open = {}                     # explanation id -> enter ts
    lect_open = {}                     # lecture id -> (enter ts, source)
    item_hist = {}                     # q -> (last answer, last correct, last q_idx, last ts)
    expl_after = collections.Counter() # q -> ms of explanation read since its last attempt
    tag_n = collections.Counter(); tag_c = collections.Counter(); tag_first = {}; tag_last = {}
    lect_n = collections.Counter(); lect_early = set(); lect_adapt = collections.Counter()
    letters = collections.Counter()
    item_seen_count = collections.Counter()  # q -> earlier attempts
    q_idx = 0

    for ts, act, item, src, ans, plat in df.itertuples(index=False):
        if t0 is None:
            t0 = ts
        if last_ts is None or ts - last_ts > SESSION_GAP_MS:
            session_idx += 1; session_pos = 0; session_start = ts
        last_ts = ts
        kind = item[0] if isinstance(item, str) and item else ""
        if act == "enter" and kind == "b":
            bundle = (item, ts, src, plat, {})
        elif act == "respond" and kind == "q" and bundle is not None and isinstance(ans, str):
            a = bundle[4].setdefault(item, [None, 0, ans, ts - bundle[1]])   # last, n, first, ms to first
            a[0] = ans; a[1] += 1
        elif act == "submit" and kind == "b" and bundle is not None:
            bid, t_enter, bsrc, bplat, answers = bundle
            elapsed = ts - t_enter
            for qid, (letter, n_resp, first_letter, t_first) in answers.items():
                info = qinfo.get(qid)
                if info is None or letter is None:
                    continue
                correct_letter, part, tags, eid, _ = info
                correct = float(letter == correct_letter)
                prev = item_hist.get(qid)
                tot_letters = sum(letters.values())
                fav = letters.most_common(1)[0][0] if letters else None
                n_t = [tag_n[t] for t in tags]
                firsts = [tag_first[t] for t in tags if t in tag_first]
                gaps = [q_idx - tag_last[t] for t in tags if t in tag_last]
                rows.append(dict(
                    user=user, q_idx=q_idx, question_id=qid, bundle_id=bid, part=part, n_tags=len(tags),
                    source=bsrc, platform=bplat, answer=letter, correct=correct,
                    elapsed_ms=float(elapsed), n_q_bundle=len(answers), n_responds=n_resp,
                    first_answer=first_letter, first_correct=float(first_letter == correct_letter),
                    t_first_ms=float(t_first),
                    first_same_as_prev=(float(first_letter == prev[0]) if prev else None),
                    item_seen=(0 if prev is None else item_seen_count[qid]),
                    prev_answer=(prev[0] if prev else None), prev_correct=(prev[1] if prev else None),
                    same_as_prev=(float(letter == prev[0]) if prev else None),
                    rep_wrong=(float(letter == prev[0]) if prev and prev[1] == 0.0 else None),
                    item_gap_q=(q_idx - prev[2] if prev else None),
                    item_gap_h=((t_enter - prev[3]) / 3.6e6 if prev else None),
                    expl_after_prev_ms=(float(expl_after[qid]) if prev else None),
                    tag_seen=(max(n_t) if n_t else 0),
                    tag_acc=(sum(tag_c[t] for t in tags) / sum(n_t) if n_t and sum(n_t) else None),
                    tag_first_correct=(sum(firsts) / len(firsts) if firsts else None),
                    tag_gap_q=(min(gaps) if gaps else None),
                    lect_before=sum(lect_n[t] for t in tags),
                    lect_early=float(any(t in lect_early for t in tags)),
                    lect_adaptive=sum(lect_adapt[t] for t in tags),
                    letter_logp=math.log((letters[letter] + 1) / (tot_letters + 4)),
                    letter_fav=(float(letter == fav) if tot_letters >= MIN_LETTERS else None),
                    session_idx=session_idx, session_pos=session_pos,
                    session_min=(t_enter - session_start) / 60000.0,
                    ts_rel_h=(t_enter - t0) / 3.6e6,
                ))
                # update histories after the row is written (no peeking)
                item_seen_count[qid] += 1
                item_hist[qid] = (letter, correct, q_idx, t_enter)
                expl_after[qid] = 0
                for t in tags:
                    if t not in tag_first:
                        tag_first[t] = correct
                    tag_n[t] += 1; tag_c[t] += correct; tag_last[t] = q_idx
                letters[letter] += 1
                q_idx += 1; session_pos += 1
            bundle = None
        elif act == "enter" and kind == "e":
            expl_open[item] = ts
        elif act == "quit" and kind == "e" and item in expl_open:
            t_in = expl_open.pop(item)
            dur = ts - t_in
            if events is not None:
                events.append(dict(user=user, kind="explanation", item_id=item, source=src, q_idx=q_idx,
                                   ts_rel_h=(t_in - t0) / 3.6e6, duration_ms=float(dur), tags=None))
            for qid in expl_to_q.get(item, []):
                if qid in item_hist:
                    expl_after[qid] += dur
        elif act == "quit" and kind == "l" and item in lect_open:
            t_in, lsrc = lect_open.pop(item)
            if events is not None:
                events.append(dict(user=user, kind="lecture", item_id=item, source=lsrc, q_idx=q_idx,
                                   ts_rel_h=(t_in - t0) / 3.6e6, duration_ms=float(ts - t_in),
                                   tags=";".join(linfo.get(item, ()))))
        elif act == "enter" and kind == "l":
            lect_open[item] = (ts, src)
            for t in linfo.get(item, ()):
                lect_n[t] += 1
                if tag_n[t] == 0:
                    lect_early.add(t)
                if isinstance(src, str) and "adaptive" in src:
                    lect_adapt[t] += 1
    return rows


def run(kt3_zip, questions_csv, lectures_csv, out, n_students, min_q, seed=1):
    import pyarrow as pa
    import pyarrow.parquet as pq
    qinfo, linfo, expl_to_q = load_contents(questions_csv, lectures_csv)
    zf = zipfile.ZipFile(kt3_zip)
    names = [n for n in zf.namelist() if n.endswith(".csv")]
    random.Random(seed).shuffle(names)
    print(f"students in zip: {len(names):,}", file=sys.stderr)
    writer, kept, total_rows, batch = None, 0, 0, []
    all_events = []
    for i, name in enumerate(names):
        if kept >= n_students:
            break
        user = name.rsplit("/", 1)[-1].replace(".csv", "")
        df = pd.read_csv(io.BytesIO(zf.read(name)))
        df = df.sort_values("timestamp", kind="stable")
        df = df[["timestamp", "action_type", "item_id", "source", "user_answer", "platform"]]
        ev = []
        rows = process_student(user, df, qinfo, linfo, expl_to_q, ev)
        if len(rows) < min_q:
            continue
        all_events.extend(ev)
        kept += 1
        batch.extend(rows)
        if len(batch) >= 200_000:
            t = pa.Table.from_pylist(batch)
            writer = writer or pq.ParquetWriter(out, t.schema)
            writer.write_table(t.cast(writer.schema)); total_rows += len(batch); batch = []
            print(f"{kept:,} students kept ({i + 1:,} read), {total_rows:,} answers", file=sys.stderr, flush=True)
    if batch:
        t = pa.Table.from_pylist(batch)
        writer = writer or pq.ParquetWriter(out, t.schema)
        writer.write_table(t.cast(writer.schema)); total_rows += len(batch)
    if writer:
        writer.close()
    ev_out = out.replace(".parquet", "_events.parquet")
    pd.DataFrame(all_events, columns=["user", "kind", "item_id", "source", "q_idx", "ts_rel_h",
                                      "duration_ms", "tags"]).to_parquet(ev_out, index=False)
    print(f"students kept: {kept:,}; answers: {total_rows:,}; explanation/lecture views: {len(all_events):,}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("kt3_zip"); ap.add_argument("questions_csv"); ap.add_argument("lectures_csv")
    ap.add_argument("--out", default="answers.parquet")
    ap.add_argument("--n-students", type=int, default=20000)
    ap.add_argument("--min-q", type=int, default=100)
    a = ap.parse_args()
    run(a.kt3_zip, a.questions_csv, a.lectures_csv, a.out, a.n_students, a.min_q)


In [ ]:
# Download KT3 and Contents (or use copies in My Drive/ednet-private/raw/ or shortcuts in My Drive)
def find_in_drive(pattern):
    hits = glob.glob(f"{DEST}/raw/{pattern}") + glob.glob(f"/content/drive/MyDrive/{pattern}")
    return hits[0] if hits else None

def fetch(short, out, pattern):
    if os.path.exists(out) and os.path.getsize(out) > 1000:
        return out
    local = find_in_drive(pattern)
    if local:
        print("using Drive copy:", local); return local
    url = requests.head(f"http://bit.ly/{short}", allow_redirects=True, timeout=30).url
    print("downloading", short, "->", url)
    gdown.download(url, out, quiet=False, fuzzy=True)
    return out

content = fetch("ednet-content", "contents.zip", "*ontent*.zip")
if not CONTENTS_ONLY:
    kt3 = fetch("ednet-kt3", "KT3.zip", "*KT3*.zip")
    print(os.path.getsize(kt3) / 1e9, "GB")
with zipfile.ZipFile(content) as z:
    z.extractall("contents")
QCSV = glob.glob("contents/**/questions.csv", recursive=True)[0]
LCSV = glob.glob("contents/**/lectures.csv", recursive=True)[0]
print(QCSV, LCSV)
if CONTENTS_ONLY:
    import shutil
    for f in (QCSV, LCSV):
        shutil.copy(f, DEST)
    print("saved to", DEST, ":", os.path.basename(QCSV), os.path.basename(LCSV), "- stopping here (CONTENTS_ONLY = True)")
    raise SystemExit("Contents saved. Done.")

In [ ]:
# One row per answered question for a random sample of students
!python ednet_pipeline.py {kt3} {QCSV} {LCSV} --out {OUT}.parquet --n-students {N_STUDENTS} --min-q {MIN_Q}

In [ ]:
# Totals only, then save to the private Drive folder (split into parts of about 25 MB by student)
import math, zlib, shutil
import pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
t = pq.read_table(f"{OUT}.parquet")
codes = pa.array([student_code(u) for u in t.column("user").to_pylist()])
t = t.set_column(t.schema.get_field_index("user"), "user", codes)   # EdNet numbers -> anonymous codes
pq.write_table(t, f"{OUT}.parquet")
ev = pq.read_table(f"{OUT}_events.parquet")
ev = ev.set_column(ev.schema.get_field_index("user"), "user",
                   pa.array([student_code(u) for u in ev.column("user").to_pylist()]))
pq.write_table(ev, f"{OUT}_events.parquet")
shutil.copy(f"{OUT}_events.parquet", DEST)
print(f"explanation/lecture views: {ev.num_rows:,}")
df = t.select(["user", "correct", "same_as_prev", "rep_wrong"]).to_pandas()
print(f"students {df.user.nunique():,}, answers {len(df):,}, accuracy {df.correct.mean():.3f}, "
      f"repeated items {df.same_as_prev.notna().mean():.3f}, repeat-wrong rate {df.rep_wrong.mean():.3f}")
size_mb = os.path.getsize(f"{OUT}.parquet") / 1e6
n_parts = max(1, math.ceil(size_mb / 25))
saved = [f"{OUT}_events.parquet"]
if n_parts == 1:
    shutil.copy(f"{OUT}.parquet", DEST); saved.append(f"{OUT}.parquet")
else:
    part = pa.array([zlib.crc32(u.encode()) % n_parts for u in t.column("user").to_pylist()])
    for k in range(n_parts):
        name = f"{OUT}_part{k + 1}of{n_parts}.parquet"
        pq.write_table(t.filter(pc.equal(part, k)), name)
        shutil.copy(name, DEST); saved.append(name)
print("saved to private Drive folder:", DEST)
for f in saved: print("  ", f, f"{os.path.getsize(os.path.join(DEST, f)) / 1e6:.1f} MB")